#### Cella import dei percorsi


In [ ]:
import sys
sys.path.append("..")
import numpy as np
from pathlib import Path
from helpers import load_csv_data

data_path = Path('../data')
cache_path = data_path / 'cache'
cache_path.mkdir(exist_ok=True)   # crea la cartella se non esiste

### Caricamento con cache

In [ ]:
if (cache_path / 'x_train.npy').exists():
    # Già salvati: li leggo veloce
    x_train = np.load(cache_path / 'x_train.npy')
    x_test = np.load(cache_path / 'x_test.npy')
    y_train = np.load(cache_path / 'y_train.npy')
    train_ids = np.load(cache_path / 'train_ids.npy')
    test_ids = np.load(cache_path / 'test_ids.npy')
    print("Caricati dalla cache")
else:
    # Prima volta: leggo i CSV (lento, 1-2 minuti) e salvo
    x_train, x_test, y_train, train_ids, test_ids = load_csv_data(str(data_path))
    np.save(cache_path / 'x_train.npy', x_train)
    np.save(cache_path / 'x_test.npy', x_test)
    np.save(cache_path / 'y_train.npy', y_train)
    np.save(cache_path / 'train_ids.npy', train_ids)
    np.save(cache_path / 'test_ids.npy', test_ids)
    print("Letti dai CSV e salvati in cache")

### Header

In [ ]:
with open(data_path / 'x_train.csv', mode='r', encoding='utf-8') as f:
    headers = f.readline().strip().split(',')
headers = headers[1:]   # tolgo 'Id'

In [ ]:
print(x_train.shape, x_test.shape, y_train.shape)
print(len(headers))

## Iniziamo a togliere alcuni valori dalla tabella

Dato che 322 valori sono troppi e molte sono ingormazioni inutili, possiamo scartare colonne come:
Id, _STATE, FMONTH, IDATE, IMONTH, IDAY, IYEAR, STATERES

Amministrative (come è stata gestita la chiamata)
DISPCODE, SEQNO, _PSU, CTELENUM, PVTRESD1, CELLFON3, NUMHHOL2, CPDEMO1, QSTVER, QSTLANG, MSCODE

Pesi statistici del sondaggio (servono ai ricercatori per ricalibrare il campione, non descrivono la persona)
_STSTR, _STRWT, _RAWRAKE, _WT2RAKE, _DUALUSE, _DUALCOR, _LLCPWT

In [ ]:
da_togliere = []

da_togliere = [
    # Identificativi e date
    'Id', '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'STATERES',
    
    # Amministrative (gestione della chiamata)
    'DISPCODE', 'SEQNO', '_PSU', 'CTELENUM', 'PVTRESD1', 'CELLFON3', 
    'NUMHHOL2', 'CPDEMO1', 'QSTVER', 'QSTLANG', 'MSCODE',
    
    # Pesi statistici del sondaggio
    '_STSTR', '_STRWT', '_RAWRAKE', '_WT2RAKE', '_DUALUSE', '_DUALCOR', '_LLCPWT'
]

# lista degli header da tenere
# scorre la lista headers, e ottiene sia l'inidice i che il nome, tiene solo gli indici in cui il nome non è nella lista da togliere 
keep = [i for i, name in enumerate(headers) if name not in da_togliere]

# selezioniamo le colonne, ovvero prendiamo tutte le righe " : ", ma solo con le colonne i cui indici sono specificati nella lista keep creata al primo passaggio.
x_train_clean = x_train[:, keep]

#Crea una nuova lista di nomi (names) che corrisponde esattamente alle colonne rimaste in x.
names = [headers[i] for i in keep]

# Applichiamo a x_test: 
x_test_clean = x_test[:, keep]

print(x_train_clean.shape, x_test_clean.shape, len(names))

In [ ]:
print(names)

In [ ]:
def informativita_colonne(x, y, names, max_cat=20, n_bin=10, top=None):
    """
    x: array 2D (con NaN), y: target in {-1, 1}, names: nomi colonne.
    Per ogni colonna calcola:
      pct_nan    : % di NaN
      rate_nan   : % di infarti tra i NaN
      rate_val   : % di infarti tra chi ha risposto
      info_nan   : |rate_nan - rate_val|  (il NaN è informativo?)
      spread     : quanto cambia il tasso di infarto tra i valori (i valori sono informativi?)
      pos_nan, pos_val: n. di infarti nei due gruppi (per capire se è rumore)
    """
    y01 = (y == 1).astype(float)
    righe = []

    for j, name in enumerate(names):
        col = x[:, j]
        nan_mask = np.isnan(col)
        n_nan = nan_mask.sum()
        n_val = len(col) - n_nan
        if n_val == 0:
            continue

        # 1) informazione del NaN
        rate_val = y01[~nan_mask].mean()
        rate_nan = y01[nan_mask].mean() if n_nan > 0 else np.nan
        info_nan = abs(rate_nan - rate_val) if n_nan > 0 else 0.0

        # 2) informazione dei valori, solo tra chi ha risposto
        v = col[~nan_mask]
        yv = y01[~nan_mask]
        if len(np.unique(v)) > max_cat:
            # colonna continua: la spezzo in n_bin gruppi di uguale numerosità
            bordi = np.unique(np.quantile(v, np.linspace(0, 1, n_bin + 1)[1:-1]))
            gruppi = np.searchsorted(bordi, v, side="right")
        else:
            _, gruppi = np.unique(v, return_inverse=True)

        n_g = np.bincount(gruppi)
        pos_g = np.bincount(gruppi, weights=yv)
        ok = n_g > 0
        rate_g = pos_g[ok] / n_g[ok]
        spread = np.sum(n_g[ok] / n_val * np.abs(rate_g - rate_val))

        righe.append((name, 100 * n_nan / len(col), 100 * rate_nan, 100 * rate_val,
                      100 * info_nan, 100 * spread,
                      int(y01[nan_mask].sum()), int(yv.sum())))

    righe.sort(key=lambda r: max(r[4], r[5]), reverse=True)
    if top:
        righe = righe[:top]

    print(f"{'colonna':<12}{'%NaN':>7}{'rate_nan':>10}{'rate_val':>10}"
          f"{'info_nan':>10}{'spread':>8}{'pos_nan':>9}{'pos_val':>9}")
    for r in righe:
        print(f"{r[0]:<12}{r[1]:7.1f}{r[2]:10.1f}{r[3]:10.1f}{r[4]:10.1f}{r[5]:8.1f}{r[6]:9d}{r[7]:9d}")
    return righe

In [ ]:
res = informativita_colonne(x_train_clean, y_train, names)
print(res)